In [167]:
import pandas as pd

daily_return_df = pd.read_csv('sp500_joined_daily_returns_yahoo.csv')
daily_return_df.set_index('Date', inplace=True)
daily_return_df

,MMM,ABT,ABBV,ABMD,ACN,ATVI,ADBE,AMD,AAP,AES,...,XEL,XRX,XLNX,XYL,YUM,ZBRA,ZBH,ZION,ZTS,SPY
Date,,,,,,,,,,,,,,,,,,,,,
2000-01-03,-0.420879,-0.395606,NaN,-0.014346,NaN,-0.064721,-0.031990,0.035491,NaN,-0.255485,...,-0.608536,-0.304356,-0.310023,NaN,-0.283451,-0.034815,NaN,-0.322457,NaN,-0.335147
2000-01-04,-0.424806,-0.399022,NaN,-0.037162,NaN,-0.114259,-0.053371,-0.033058,NaN,-0.256409,...,-0.587705,-0.358993,-0.290556,NaN,-0.268203,-0.014428,NaN,-0.303260,NaN,-0.340144
2000-01-05,-0.396778,-0.384488,NaN,0.054945,NaN,-0.093212,0.044063,0.032258,NaN,-0.224806,...,-0.580057,-0.291187,-0.297410,NaN,-0.259446,0.066588,NaN,-0.279348,NaN,-0.321987
2000-01-06,-0.370289,-0.371148,NaN,0.023050,NaN,-0.114423,-0.002024,0.032258,NaN,-0.228983,...,-0.598170,-0.322374,-0.341881,NaN,-0.260714,-0.041970,NaN,-0.269102,NaN,-0.331391
2000-01-07,-0.401052,-0.373645,NaN,-0.003472,NaN,-0.089726,0.044336,0.054767,NaN,-0.214733,...,-0.595659,-0.311802,-0.214726,NaN,-0.275959,-0.006455,NaN,-0.281024,NaN,-0.296027
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-11-30,-0.017295,0.004176,-0.005137,0.001388,0.000120,0.014811,-0.003187,0.061033,-0.010915,-0.025274,...,-0.003698,-0.047018,0.058622,-0.008779,-0.003766,-0.000159,-0.007455,-0.019563,-0.012925,-0.002122
2020-12-01,-0.021237,-0.006866,-0.009614,-0.014332,0.003022,-0.000125,-0.005934,0.004119,0.023747,-0.002904,...,0.011683,-0.006278,0.005517,-0.004614,-0.004680,-0.024788,-0.029171,0.013661,-0.001732,0.001231
2020-12-02,0.009339,-0.001477,0.009519,-0.011073,-0.004188,-0.007150,0.011752,0.009151,-0.018058,0.033203,...,0.007334,0.014324,0.008985,-0.000935,-0.008407,-0.009441,0.017130,0.040000,-0.012733,0.005400


### Follow The Leader ###

In [ ]:
import numpy as np

windows = [5,10,20,40,60]
W = pd.DataFrame().reindex(columns = daily_return_df.columns, index = windows)
for col in W.columns:
    W[col].values[:] = 0
FTL_betas = pd.DataFrame().reindex(columns = daily_return_df.columns, index = daily_return_df.index)
for date in daily_return_df.index:
    print("W on {1}: {0}".format(W, date))
    print("W.idxmin(): {0}".format(W.idxmin()))
    min_window_df = W.idxmin()
    print("min window: {0}".format(min_window_df))
    for ticker in daily_return_df.columns:
        window_for_ticker = min_window_df[ticker]
        act_beta = daily_return_df[ticker].rolling(window=window_for_ticker).cov(daily_return_df['SPY']).div(daily_return_df['SPY'].rolling(window=window_for_ticker).var(), axis=0)
        FTL_betas[ticker][date] = act_beta[date]
        
        # refresh weights
        act_beta_error = (daily_return_df[ticker] - act_beta.mul(daily_return_df['SPY'].mean(), axis=0)) ** 2
        
        if not np.isnan(act_beta_error[date]):
#             print("act beta: {0}".format(act_beta_error[date]))
#             print(":::::::::::::W[ticker][window_for_ticker]: {}".format(W[ticker][window_for_ticker]))
            W[ticker][window_for_ticker] += act_beta_error[date]
#             print(":::::::::::::W[ticker][window_for_ticker]: {}".format(W[ticker][window_for_ticker]))
#         if ticker == "ABBV":
#             print("ABBV ==================")
#             print(date)
#             print(act_beta_error)
#             print(W[ticker])
#             print("ABBV ==================")
            
#         else:
#             print(ticker)
        
        for window in windows:
            if window == window_for_ticker:
                continue
            beta = daily_return_df[ticker].rolling(window=window).cov(daily_return_df['SPY']).div(daily_return_df['SPY'].rolling(window=window).var(), axis=0)
            beta_error = (daily_return_df[ticker] - beta.mul(daily_return_df['SPY'].mean(), axis=0)) ** 2
#             if not np.isnan(beta_error[date]):
#                 W[ticker] += beta_error[date]
    print("//////////////////////////////////")

W on 2000-01-03:     MMM  ABT  ABBV  ABMD  ACN  ATVI  ADBE  AMD  AAP  AES  ...  XEL  XRX  XLNX  \
5   0.0  0.0   0.0   0.0  0.0   0.0   0.0  0.0  0.0  0.0  ...  0.0  0.0   0.0   
10  0.0  0.0   0.0   0.0  0.0   0.0   0.0  0.0  0.0  0.0  ...  0.0  0.0   0.0   
20  0.0  0.0   0.0   0.0  0.0   0.0   0.0  0.0  0.0  0.0  ...  0.0  0.0   0.0   
40  0.0  0.0   0.0   0.0  0.0   0.0   0.0  0.0  0.0  0.0  ...  0.0  0.0   0.0   
60  0.0  0.0   0.0   0.0  0.0   0.0   0.0  0.0  0.0  0.0  ...  0.0  0.0   0.0   

    XYL  YUM  ZBRA  ZBH  ZION  ZTS  SPY  
5   0.0  0.0   0.0  0.0   0.0  0.0  0.0  
10  0.0  0.0   0.0  0.0   0.0  0.0  0.0  
20  0.0  0.0   0.0  0.0   0.0  0.0  0.0  
40  0.0  0.0   0.0  0.0   0.0  0.0  0.0  
60  0.0  0.0   0.0  0.0   0.0  0.0  0.0  

[5 rows x 506 columns]
W.idxmin(): MMM     5
ABT     5
ABBV    5
ABMD    5
ACN     5
       ..
ZBRA    5
ZBH     5
ZION    5
ZTS     5
SPY     5
Length: 506, dtype: int64
min window: MMM     5
ABT     5
ABBV    5
ABMD    5
ACN     5
       ..


//////////////////////////////////
W on 2000-01-11:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE      AMD  AAP  \
5   0.119369  0.084679   0.0  0.007813  0.0  0.002548  0.177506  0.08509  0.0   
10  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.00000  0.0   
20  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.00000  0.0   
40  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.00000  0.0   
60  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.00000  0.0   

         AES  ...       XEL       XRX     XLNX  XYL       YUM      ZBRA  ZBH  \
5   0.002621  ...  0.342175  0.039387  0.03338  0.0  0.083726  0.014248  0.0   
10  0.000000  ...  0.000000  0.000000  0.00000  0.0  0.000000  0.000000  0.0   
20  0.000000  ...  0.000000  0.000000  0.00000  0.0  0.000000  0.000000  0.0   
40  0.000000  ...  0.000000  0.000000  0.00000  0.0  0.000000  0.000000  0.0   
60  0.000000  ...  0.000000  0.000000  0.00000  0.0  0.000000

//////////////////////////////////
W on 2000-01-19:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.119369  0.084679   0.0  0.007813  0.0  0.002548  0.177506  0.085090   
10  0.135212  0.098252   0.0  0.044857  0.0  0.002793  0.171194  0.119554   
20  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   
40  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   
60  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.002621  ...  0.342175  0.039387  0.033380  0.0  0.083726  0.014248   
10  0.0  0.001301  ...  0.349649  0.061620  0.020995  0.0  0.079745  0.092313   
20  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   
40  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   
60  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   

//////////////////////////////////
W on 2000-01-26:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.119369  0.084679   0.0  0.007813  0.0  0.002548  0.177506  0.085090   
10  0.135212  0.098252   0.0  0.044857  0.0  0.002793  0.171194  0.119554   
20  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   
40  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   
60  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.002621  ...  0.342175  0.039387  0.033380  0.0  0.083726  0.014248   
10  0.0  0.001301  ...  0.349649  0.061620  0.020995  0.0  0.079745  0.092313   
20  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   
40  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   
60  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   

//////////////////////////////////
W on 2000-02-02:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.119369  0.084679   0.0  0.007813  0.0  0.002548  0.177506  0.085090   
10  0.135212  0.098252   0.0  0.044857  0.0  0.002793  0.171194  0.119554   
20  0.097757  0.102681   0.0  0.002758  0.0  0.031448  0.051833  0.227827   
40  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   
60  0.000000  0.000000   0.0  0.000000  0.0  0.000000  0.000000  0.000000   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.002621  ...  0.342175  0.039387  0.033380  0.0  0.083726  0.014248   
10  0.0  0.001301  ...  0.349649  0.061620  0.020995  0.0  0.079745  0.092313   
20  0.0  0.019843  ...  0.354362  0.050034  0.007615  0.0  0.061583  0.002325   
40  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   
60  0.0  0.000000  ...  0.000000  0.000000  0.000000  0.0  0.000000  0.000000   

...

//////////////////////////////////
W on 2000-05-05:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.536570  0.697195   0.0  0.763110  0.0  5.668286  1.909210  0.356380   
10  0.632086  0.757922   0.0  0.618837  0.0  0.187910  1.673824  0.360829   
20  0.522878  0.714455   0.0  0.234675  0.0  0.098574  1.625141  0.457183   
40  0.599544  0.749070   0.0  0.199367  0.0  0.132909  1.757077  0.345141   
60  0.550527  0.784277   0.0  0.201519  0.0  0.096199  1.551447  0.383722   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.304609  ...  2.287761  0.425937  0.148585  0.0  0.083726  0.162268   
10  0.0  0.360758  ...  2.053395  0.306598  0.206039  0.0  0.079745  0.185635   
20  0.0  0.279893  ...  2.056506  0.215610  0.206856  0.0  0.074229  0.180582   
40  0.0  0.258058  ...  2.226240  0.223192  0.134687  0.0  0.069319  0.131290   
60  0.0  0.264056  ...  2.189319  0.228441  0.122682  0.0  0.081634  0.149303   

//////////////////////////////////
W on 2000-05-12:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.634065  0.959663   0.0  0.763110  0.0  5.668286  2.184336  0.872812   
10  0.740078  0.858348   0.0  0.618837  0.0  0.187910  2.808797  0.509076   
20  0.635523  0.801620   0.0  0.234675  0.0  0.122331  2.137589  0.457183   
40  0.698503  0.859527   0.0  0.286281  0.0  0.132909  2.208057  0.455017   
60  0.642298  0.784277   0.0  0.223411  0.0  0.103114  1.934219  0.435864   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.327184  ...  2.671769  0.425937  0.148585  0.0  0.083726  0.502849   
10  0.0  0.360758  ...  2.379738  0.306598  0.206039  0.0  0.079745  0.185635   
20  0.0  0.325095  ...  2.388825  0.335281  0.206856  0.0  0.081325  0.180582   
40  0.0  0.332429  ...  2.555207  0.296876  0.171147  0.0  0.076192  0.261921   
60  0.0  0.317220  ...  2.507318  0.259172  0.168329  0.0  0.081634  0.175532   

//////////////////////////////////
W on 2000-05-19:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.788311  0.959663   0.0  0.763110  0.0  5.668286  2.291203  0.872812   
10  0.740078  0.968322   0.0  0.618837  0.0  0.187910  2.808797  0.622951   
20  0.772559  0.905857   0.0  0.274723  0.0  0.122331  2.754940  0.737961   
40  0.815910  0.975358   0.0  0.286281  0.0  0.132909  2.706032  0.564983   
60  0.817628  0.979909   0.0  0.268946  0.0  0.111890  2.274135  0.532702   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.526263  ...  3.031577  0.425937  0.260231  0.0  0.258106  0.502849   
10  0.0  0.360758  ...  2.748960  0.380278  0.301885  0.0  0.148340  0.353260   
20  0.0  0.358445  ...  2.730166  0.335281  0.206856  0.0  0.093061  0.323533   
40  0.0  0.350393  ...  2.907775  0.337230  0.264295  0.0  0.107582  0.261921   
60  0.0  0.337876  ...  2.834740  0.354779  0.235370  0.0  0.085640  0.242467   

//////////////////////////////////
W on 2000-05-26:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.955574  1.729862   0.0  0.763110  0.0  5.668286  7.240056  0.872812   
10  0.856708  1.171461   0.0  0.618837  0.0  0.187910  2.808797  0.710107   
20  0.912901  1.043817   0.0  0.355543  0.0  0.122331  2.913773  0.737961   
40  0.916273  1.090748   0.0  0.349974  0.0  0.132909  3.143382  0.710380   
60  0.916638  1.091238   0.0  0.338639  0.0  0.122475  3.075938  0.737523   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.526263  ...  3.236577  0.425937  0.910756  0.0  0.258106  0.502849   
10  0.0  0.380678  ...  3.119041  0.466770  0.301885  0.0  0.148340  0.353260   
20  0.0  0.372830  ...  3.061052  0.399158  0.290043  0.0  0.108559  0.474633   
40  0.0  0.373159  ...  3.242268  0.392822  0.411980  0.0  0.107582  0.345735   
60  0.0  0.388539  ...  3.171247  0.422217  0.328837  0.0  0.107417  0.354976   

//////////////////////////////////
W on 2000-06-05:          MMM       ABT  ABBV      ABMD  ACN      ATVI      ADBE       AMD  \
5   0.955574  1.729862   0.0  0.763110  0.0  5.668286  7.240056  0.872812   
10  0.956320  1.384174   0.0  0.618837  0.0  0.187910  3.373141  0.989728   
20  0.994466  1.375141   0.0  0.371008  0.0  0.194494  3.312101  0.926597   
40  1.033400  1.219025   0.0  0.409483  0.0  0.132909  3.143382  1.028991   
60  1.024350  1.214278   0.0  0.379452  0.0  0.139164  3.489130  0.809691   

    AAP       AES  ...       XEL       XRX      XLNX  XYL       YUM      ZBRA  \
5   0.0  0.526263  ...  3.478708  0.560210  0.910756  0.0  0.258106  0.502849   
10  0.0  0.429190  ...  3.425780  0.466770  0.438756  0.0  0.148340  0.935982   
20  0.0  0.388533  ...  3.363860  0.479916  0.406965  0.0  0.154627  0.474633   
40  0.0  0.398950  ...  3.585641  0.456242  0.411980  0.0  0.128174  0.486480   
60  0.0  0.388539  ...  3.503664  0.496071  0.389344  0.0  0.115857  0.512868   

### Random Forest ###

In [ ]:
import numpy as np

windows = [5,10,20,40,60]
min_windows = pd.DataFrame().reindex_like(daily_return_df)
concat = None
for w in windows:
    beta_df = daily_return_df.rolling(window=w).cov(daily_return_df['SPY']).div(daily_return_df['SPY'].rolling(window=w).var(), axis=0)
    act_beta_error_df = (daily_return_df - beta_df.mul(daily_return_df['SPY'].mean(), axis=0)) ** 2    
    act_beta_error_df.columns = act_beta_error_df.columns + "_{0}".format(w)
    if concat is not None:
        concat = pd.concat([concat,act_beta_error_df], axis=1)
    else:
        concat = act_beta_error_df
concat

In [378]:
min_windows = pd.DataFrame()
for ticker in daily_return_df.columns:
    prefix = ticker + '_'
    head = [prefix + "5", prefix + "10", prefix + "20", prefix + "40", prefix + "60"]
    min_windows[ticker] = concat[head].idxmin(axis=1).str.replace(prefix,"")
min_windows

,MMM,ABT,ABBV,ABMD,ACN,ATVI,ADBE,AMD,AAP,AES,...,XEL,XRX,XLNX,XYL,YUM,ZBRA,ZBH,ZION,ZTS,SPY
Date,,,,,,,,,,,,,,,,,,,,,
1/2/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1/3/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1/6/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1/7/2020,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1/8/2020,5,5,5,5,5,5,5,5,5,5,...,5,5,5,5,5,5,5,5,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11/20/2020,10,5,40,20,10,5,40,10,5,5,...,10,5,10,5,20,40,5,5,5,5
11/23/2020,20,20,5,20,10,20,40,10,40,5,...,5,10,10,5,5,40,10,5,10,40
11/24/2020,20,20,5,20,10,20,20,10,40,20,...,40,10,10,10,10,5,40,5,10,5


In [379]:

d = pd.DataFrame(np.zeros((df.shape)), index=df.index, columns=df.columns)

cumsum_df = daily_return_df.cumsum(axis=0) #vegtelen ablak
print(cumsum_df)
# cumsum_df = daily_return_df.rolling(window=100) #rolling min (nem az egeszen, csak pl 100 napon)
# # melyikre teljes\t jobban? fel-1-20... evek vagy a vegtelen
# # follow the leader x ev
# # ftl vegtelen, stb.

# #cumsum.idxmin(axis=1)
# #print(d)
# for ind,row in cumsum_df.idxmin(axis=1).items():
#     #print(ind)
#     #print(row)
#     d.loc[ind,row] = 1

# shifted = d.shift(1) # Mert a tegnapig kapott adatokat nezzuk
# shifted = shifted.mul(df)
# print(df.tail())
# print(df.shift(1).tail())
# shifted.sum(axis=1).cumsum().plot()
# cumsum_df["follow the leader"] = shifted.sum(axis=1).cumsum()
# cumsum_df

# # A mai legjobbat a holnapi legjobbal kell szamolni: shift 1

# # Feature set a kapott beta ablakokkal minden evre, minden modell csak a sajat evet nezi
# # melyik nap 
# # volatilis iszoszakok: beta ablak
# # X: amit felhasznalok feature, Y: a kimenet
# # X-et kitalalom pl. az elmult valamennyi nap alapjan a tulajdonsagok
# # minden napra visszamegyek egy param pl 5 napot, es 
# # daily return, szoras
# # beta window melle a volume


# # kiszamolni minden reszvenyre egy napon belul a legjobb window
# # minden reszveny, datum-ra minden window, es a legkisebbet veszem
# # shift ide nem kell


                 MMM       ABT      ABBV      ABMD       ACN      ATVI  \
Date                                                                     
1/2/2020    0.013057  0.010342  0.005276 -0.021448 -0.003367 -0.021032   
1/3/2020    0.021135  0.011974  0.011287 -0.021508  0.002868 -0.010003   
1/6/2020    0.029433  0.020383  0.024663  0.053614  0.003636  0.011718   
1/7/2020    0.027302  0.021433  0.024213  0.062113  0.007229  0.022607   
1/8/2020    0.042077  0.021085  0.028363  0.063851  0.008846  0.020607   
...              ...       ...       ...       ...       ...       ...   
11/20/2020 -0.229102  0.119007 -0.080462  0.670294  0.258689  0.215344   
11/23/2020 -0.226915  0.099622 -0.078108  0.664065  0.264778  0.202343   
11/24/2020 -0.215029  0.084070 -0.067610  0.661175  0.275247  0.193124   
11/25/2020 -0.213446  0.078092 -0.063950  0.654175  0.272567  0.200763   
11/27/2020 -0.218676  0.086998 -0.064808  0.684099  0.272726  0.206426   

                ADBE       AMD       

AttributeError: 'Rolling' object has no attribute 'idxmin'

In [ ]:
cumsum_df

In [ ]:
# X feature 
# shift-el veszem az elozo x napot, 
pd.DataFrame({"returns" : daily_return_df.stack(), "shifted returns": daily_return_df.shift(1).stack()})
# X feature volume, rolling 
# 

In [ ]:
daily_return_df.rolling(10).std()
# szoras, vagy 
daily_return_df.rolling(10).mean()
# Ezekbol csinalom az X-et

# X-bol az Y-t random forest-el szamolom
# A holnapi Y-t a mai X-hez: shift -1 (indexek stimmeljenek)
# Az Y az optimalis window merete

#FTL: minden reszvenyre kell az optimalis window
# --> a kovetkezonek minek kellene lennie

# random forest: default
# szimulacio: megtanitom pl 10-ig(traning), es utana 20-ig(test) ellenorzom
from sicklearn import randomforest
# fit # predict

## step:
# ftl osszehasonlitasa a window-kra, es a vegtelenre
# beta error reszvenyenkent minden ablakra 
## ez lesz az X (+ egy szorast, meg egy arfolyamerteket min, max, volume)
## Y a beta_error reszvenyenkent
# Y minden napra minden reszvenyre kiszamolom az osszes beta window hibajat, es a legkisebb ablakja lesz az Y
# Egy napon belülre számolom az értékeket: 5 érték 

# Fix es X es Y ertekeket megetetetek a random forest-el
